# Can we change a campaign background while preserving the product, branding, and geometry?

## 1. Before You Begin

We will place the committed Adventurer Pro Backpack photograph into a
trailhead campaign scene while explicitly preserving the product. Our single
focus is **high-fidelity product-preserving editing** with
GPT-Image-2.5-Sunburst.

Deploy the model in Microsoft Foundry and configure section 2. The request is
paid and runs only when valid credentials are present and the request cell is
executed. See
[current pricing](https://azure.microsoft.com/pricing/details/azure-openai/).


In [ ]:
# 2. Verify your environment
import os
from pathlib import Path

import requests
from dotenv import load_dotenv

load_dotenv()

def find_assets() -> Path:
    """Find the shared data whether Jupyter starts here or at the repo root."""
    for base in (Path.cwd(), *Path.cwd().parents):
        candidate = base / "models/azure-openai/shared/contoso-outdoors"
        if (candidate / "products.json").is_file():
            return candidate
    raise FileNotFoundError(
        "Could not find models/azure-openai/shared/contoso-outdoors. "
        "Run this notebook from a checkout of the model-releases repository."
    )


required = ["AZURE_OPENAI_ENDPOINT", "AZURE_OPENAI_API_KEY", "AZURE_OPENAI_GPT_IMAGE_25_SUNBURST_DEPLOYMENT"]
missing = [name for name in required if not os.getenv(name)]
if missing:
    raise EnvironmentError(
        f"Missing {missing}. Copy scripts/sample.env to .env, add the values, "
        "and review models/quickstart/README.md."
    )

endpoint = os.environ["AZURE_OPENAI_ENDPOINT"].rstrip("/")
api_key = os.environ["AZURE_OPENAI_API_KEY"]
deployment = os.environ["AZURE_OPENAI_GPT_IMAGE_25_SUNBURST_DEPLOYMENT"]
assets = find_assets()
output_dir = assets.parents[1] / "gpt-image-2.5-sunburst" / "output"
output_dir.mkdir(exist_ok=True)

print(f"Environment ready for deployment: {deployment}")
print(f"Generated images will be written to: {output_dir.resolve()}")


## 3. Define what must remain unchanged

The committed product photograph is our visual source of truth. We inspect the
matching catalog record and state preservation constraints before editing:
backpack shape, blue color, straps, pockets, and visible construction.


In [ ]:
# 4. Load the backpack record and display the source image
import base64
import json

from IPython.display import HTML, Image, display

products = json.loads((assets / "products.json").read_text(encoding="utf-8"))
backpack = next(product for product in products if product["id"] == 2)
source_image = assets / backpack["images"][0]
if not source_image.is_file():
    raise FileNotFoundError(source_image)

preserve = [
    "overall backpack shape",
    "blue product color",
    "shoulder and sternum straps",
    "visible pockets and attachment points",
    "existing product construction and marks",
]
print(json.dumps({key: backpack[key] for key in ("id", "name", "brand", "category")}, indent=2))
print("Preserve:", *preserve, sep="\n- ")
encoded_source = base64.b64encode(source_image.read_bytes()).decode("ascii")
display(HTML(f'<img src="data:image/webp;base64,{encoded_source}" width="420">'))


## 5. Make one controlled campaign edit

The request states explicit preservation constraints and asks for a
background-only creative change. The output still requires human review; the
notebook does not claim pixel-perfect preservation or automatically approve
the campaign asset.


In [ ]:
# 6. Edit the setting while preserving the backpack
api_version = "2025-04-01-preview"
edit_url = (
    f"{endpoint}/openai/deployments/{deployment}/images/edits"
    f"?api-version={api_version}"
)
prompt = """
Create one outdoor-retail campaign image from this source photograph.
Preserve the Adventurer Pro Backpack itself: keep its shape, blue color,
straps, pockets, attachment points, proportions, and visible construction
unchanged. Do not add or alter logos or text on the product. Change only the
surrounding setting to a clean morning trailhead scene with soft natural light.
Do not add headline text, badges, or unrelated products.
"""

with source_image.open("rb") as image_file:
    response = requests.post(
        edit_url,
        headers={"api-key": api_key},
        data={
            "prompt": prompt,
            "n": "1",
            "size": "1024x1024",
            "quality": "high",
            "output_format": "png",
        },
        files={"image": (source_image.name, image_file, "image/webp")},
        timeout=300,
    )
if not response.ok:
    raise RuntimeError(f"Image edit failed ({response.status_code}): {response.text}")

payload = response.json()
if not payload.get("data"):
    raise RuntimeError(f"Image edit returned no image data: {payload}")
image_bytes = base64.b64decode(payload["data"][0]["b64_json"], validate=True)
if not image_bytes.startswith(b"\x89PNG\r\n\x1a\n"):
    raise ValueError("The returned image is not a PNG")

output_path = output_dir / "sunburst-backpack-campaign-edit.png"
output_path.write_bytes(image_bytes)
display(Image(data=image_bytes, width=420))
print(f"Saved: {output_path}")

# Human review remains required for these visual preservation criteria.
print("\nReview against the source:")
for criterion in preserve:
    print(f"[ ] {criterion}")
print("[ ] only the setting and lighting changed")
print("[ ] no invented branding or campaign text")


## 7. Your Turn to Explore

- Change only the time of day while retaining the same trailhead composition.
- Use the committed tent image and write a product-specific preservation list.
- Request a transparent background for a separate product-cutout workflow.


## 8. Summary

We used GPT-Image-2.5-Sunburst for one capability: a high-fidelity campaign
edit with explicit product-preservation constraints. The service produced a
candidate; a person still compares it with the approved source before use.
Review the [image-generation primer](../../../docs/primers/image-generation.md)
for broader workflow and safety considerations.


## 9. References

- [GPT-Image-2.5-Sunburst model card](https://ai.azure.com/catalog/models/gpt-image-2.5-sunburst) — Foundry catalog entry.
- [Use image generation models from OpenAI](https://learn.microsoft.com/en-us/azure/foundry/openai/how-to/dall-e) — image editing parameters and response format.
- [Use the Azure OpenAI Responses API](https://learn.microsoft.com/en-us/azure/foundry/openai/how-to/responses) — supported model version listing.
- [Create multimodal applications with OpenAI models in Microsoft Foundry](https://techcommunity.microsoft.com/blog/azure-ai-foundry-blog/create-multimodal-applications-with-openai-models-in-microsoft-foundry/4543593) — release announcement.
